In [ ]:
!pip -q install requests==2.32.4 urllib3==2.5.0 "jedi>=0.16" negmas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.2.1 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [ ]:
import importlib
importlib.reload(bridge_service)
importlib.reload(summariser_module)
importlib.reload(topic_engine)

<module 'topic_engine' from '/content/topic_engine.py'>

In [ ]:
### Keep these modules in session - bridge_service | summariser_module | topic_engine | structured_conversation.json (for local testing)
import os
import json
import time
import urllib3
import decimal
from datetime import datetime, timezone
from google.colab import userdata

import bridge_service
import summariser_module
import topic_engine
from bridge_service import make_config_from_env, MoltbookMyceliumBridge
from summariser_module import summarize_for_moltbook
from topic_engine import (
    WindowManager,
    TopicDiscoveryEngine,
    CandidateIntelligenceEngine,
    CandidateValidationEngine,
    execute_governance_gate
)

print("modules import OK")
urllib3.disable_warnings()

modules import OK


In [ ]:
# ---------- 0. Secrets ----------
_SECRET_ENV_MAP = {
    "LITELLM_API_KEY":               "LITELLM_API_KEY",
    "LITELLM_API_KEY2":              "LITELLM_API_KEY2",
    "LITELLM_USER_ID1":              "LITELLM_USER_ID1",
    "LITELLM_USER_ID2":              "LITELLM_USER_ID2",
    "MOLTBOOK_API_KEY":              "MOLTBOOK_API_KEY",
    "MOLTBOOK_API_KEY_FINANCEGUARD": "MOLTBOOK_API_KEY_FINANCEGUARD",
    "MOLTBOOK_API_KEY_CAREERMAX":    "MOLTBOOK_API_KEY_CAREERMAX",
    "MOLTBOOK_API_KEY_STUDENTLIFE":  "MOLTBOOK_API_KEY_STUDENTLIFE",
    "MOLTBOOK_API_KEY_RISKANALYST":  "MOLTBOOK_API_KEY_RISKANALYST",
}

missing = []
for secret_name, env_var in _SECRET_ENV_MAP.items():
    try:
        os.environ[env_var] = userdata.get(secret_name)
    except userdata.SecretNotFoundError:
        missing.append(secret_name)
    except userdata.NotebookAccessError:
        raise RuntimeError(
            f"Notebook not granted access to secret '{secret_name}'. "
            "Toggle 'Notebook access' ON in the Colab Secrets panel."
        )
    except Exception as e:
        print(f"⚠️ Secret '{secret_name}' not loaded: {e}")

if missing:
    raise RuntimeError(f"Add these secrets in the Colab Secrets panel: {missing}")


# ---------- 1. Core Parameters ----------
os.environ['MOLTBOOK_BASE_URL'] = 'https://www.moltbook.com/api/v1'
os.environ['MOLTBOOK_SUBMOLT'] = 'secure-dev-logic'
os.environ['MOLTBOOK_SUBMOLT_DISPLAY'] = 'Private Agent Logic Exchange'
os.environ['MOLTBOOK_SUBMOLT_DESCRIPTION'] = 'A restricted room for back-end multi-agent synchronization.'
os.environ['MYCELIUM_BASE_URL'] = 'https://argue-cant-billing-endorsed.trycloudflare.com'
os.environ["MYCELIUM_ROOM"] = "moltbook-mycelium-demo"
os.environ['MYCELIUM_SENDER'] = 'amlan'
os.environ["BRIDGE_STATE_PATH"] = "bridge_state.json"

# ---------- 2. LiteLLM Parameters ----------
os.environ["LITELLM_MODEL"] = "azure/gpt-4o-mini"
os.environ["LITELLM_URL"] = "https://litellm.prod.outshift.ai/v1/chat/completions"

# ---------- 3. Per-experiment inputs ----------
# Scenario facts are injected into each agent lens by bridge_service; override per run.
os.environ.setdefault(
    "SCENARIO_FACTS",
    "ASU has a $25,000/year scholarship and NYU has no aid.",
)
# Target post to (re)run. Optional title check runs only if EXPECTED_TOPIC_TITLE is set.
os.environ.setdefault("TARGET_POST_ID", "53fff94b-6542-4b91-bdcc-aabf9c9d237a")
os.environ.setdefault("EXPECTED_TOPIC_TITLE", "")
# Set to "True" to use structured_conversation.json instead of Live Moltbook API
os.environ.setdefault("USE_LOCAL_JSON", "True")

# ---------- 4. Setup & bootstrapping - Load config + build bridge ----------
cfg = make_config_from_env()
bridge = MoltbookMyceliumBridge(cfg)
print("###################################################")
print("\n[SYSTEM BOOTSTRAP] Parameters loaded and Bridge configured.")
print("###################################################")
print(cfg)

# ---------- 5. Bootstrapping environment checks ----------
submolt = bridge.ensure_submolt()
print("###################################################")
print("environment checks done")
print("###################################################")
print("SUBMOLT READY PROFILE:", submolt)

# ---------- 6. Smoke test Moltbook posts and Mycelium messages ----------
print(bridge.moltbook.list_submolt_posts(cfg.submolt_name, limit=5))
# print(bridge.mycelium.get_messages(cfg.mycelium_room, limit=5))

posts = bridge.moltbook.list_submolt_posts(cfg.submolt_name, limit=50)
print("###################################################")
print("list submolt posts done")
print("###################################################")

# ---------- 7. Optional post title verification ----------
expected_title = os.environ.get("EXPECTED_TOPIC_TITLE", "").strip()
if expected_title:
    print("###################################################")
    print("post name discovery done/ title verification")
    print("###################################################")
    for p in posts:
        print(p.get("id"), "|", p.get("title") or p.get("name"))
        title = (p.get("title") or p.get("name") or "").strip()
        if title.lower() == expected_title.lower():
            print("FOUND:", p)
            break
    else:
        raise ValueError(f"Expected topic title not found: {expected_title!r}")
else:
    print("NOTE: EXPECTED_TOPIC_TITLE not set — skipping title verification.")


# ---------- 8. Conversation event Stream Ingestion & Autonomous Topic Mining ----------
USE_LOCAL_JSON = os.environ.get("USE_LOCAL_JSON", "False").lower() == "true"
target_post_id = os.environ["TARGET_POST_ID"]
# target_post_id = os.environ.get("TARGET_POST_ID", "53fff94b-6542-4b91-bdcc-aabf9c9d237a")
print("###################################################")
print(f"Fetching live event stream for Thread ID: {target_post_id}...")
print("###################################################")

print("\n" + "=" * 80)
if USE_LOCAL_JSON:
    print(f"DEMO MODE: Loading static stream from 'structured_conversation.json'...")
    try:
        with open("structured_conversation.json", "r") as f:
            raw_stream = json.load(f)
        # Sort local JSON chronologically
        live_event_stream = sorted(raw_stream, key=lambda x: x.get("created_at", ""))
        print(f"[STREAM INGESTION COMPLETE] Loaded {len(live_event_stream)} events from local JSON.")
    except Exception as e:
        raise RuntimeError(f"Failed to load structured_conversation.json: {e}")
else:
    print(f"LIVE MODE: Fetching event stream from Moltbook Thread ID: {target_post_id}...")
    submolt = bridge.ensure_submolt()

    def flatten_comment_tree(comment_node, parent_id=None):
        """
        Recursively unpacks nested 'replies' into a flat list, preserving
        explicit parent_id references for downstream thread tracing.
        """
        flat = []
        # Set parent_id on the node if not present
        node_copy = dict(comment_node)
        if parent_id and not node_copy.get("parent_id"):
            node_copy["parent_id"] = parent_id

        # Extract nested replies before removing them from the flat representation
        nested_replies = node_copy.pop("replies", []) or []

        # Tag as comment event
        node_copy["post_id"] = target_post_id
        node_copy["event_type"] = "COMMENT_CREATED"
        flat.append(node_copy)

        # Recursively traverse any child replies
        current_node_id = node_copy.get("id")
        for child in nested_replies:
            flat.extend(flatten_comment_tree(child, parent_id=current_node_id))

        return flat


        try:
            # 1. Fetch Root Post
            root_post_response = bridge.moltbook._get(f"/posts/{target_post_id}")
            root_post = root_post_response.get("post", {})
            root_post["event_type"] = "POST_CREATED"
            root_post["parent_id"] = None

            # 2. Paginated Fetch across all offsets
            all_comments_raw = []
            offset = 0
            limit = 20

            while True:
                res = bridge.moltbook._get(f"/posts/{target_post_id}/comments?limit={limit}&offset={offset}")
                batch = res.get("comments", [])
                if not batch:
                    break

                all_comments_raw.extend(batch)
                print(f" ├── Fetched offset {offset}: retrieved {len(batch)} parent comments")

                has_more = res.get("has_more", False)
                if not has_more or len(batch) < limit:
                    break

                offset += limit

            # 3. Recursively flatten all parent comments and nested agent child replies
            all_flat_comments = []
            for parent_comment in all_comments_raw:
                all_flat_comments.extend(flatten_comment_tree(parent_comment, parent_id=root_post.get("id")))

            # 4. Sort strictly forward in time (Chronological Order: oldest -> newest)
            all_flat_comments.sort(key=lambda x: x.get("created_at", ""))

            # 5. Build final Chronological Event Stream (Root post first, then all comments)
            live_event_stream = [root_post] + all_flat_comments

            print(f"\n[STREAM INGESTION COMPLETE]")
            print(f"Total stream events assembled: {len(live_event_stream)} (1 root post + {len(all_flat_comments)} total hierarchical comments)")

            # Quick participant check
            authors_found = set()
            for ev in live_event_stream:
                a = ev.get("author", {})
                authors_found.add(a.get("name") if isinstance(a, dict) else str(a))
            print(f"Unique agents/authors discovered in stream: {sorted(list(authors_found))}")

        except Exception as e:
            raise RuntimeError(f"Failed to fetch live stream from Moltbook: {e}")

In [ ]:

# -------------------------------------------------------------
#  Stages 1-7: Topic Mining, Enrichment & Scoring
# -------------------------------------------------------------
window_mgr = WindowManager(window_limit=5, dedup_cache_size=500)
discovery_engine = TopicDiscoveryEngine()
intel_engine = CandidateIntelligenceEngine()
validation_engine = CandidateValidationEngine()

# Simulated system state stores for Stage 7 gatekeeper checks
system_state_store = {
    "active_missions": [],
    "open_candidates": [],
    "cooldown_history": {}
}

print("\n" + "="*80)
print("STARTING CHRONOLOGICAL TOPIC MINING, ENRICHMENT & SCORING (STAGES 1-5) ...")
print("="*80)

latest_enriched_candidates = []
latest_scored_candidates = []
final_window = None
enriched_topic_lookup = {} # Dynamic capture dict for Stage 10

for raw_event in live_event_stream:
    time.sleep(1) # Slight pause to stagger LLM API requests

    # S1 & S2: Filter, Normalize, and Update Hybrid Window
    current_window = window_mgr.process_event(raw_event)
    final_window = current_window

    raw_id = raw_event.get("id") or raw_event.get("message_id")
    if not current_window:
        print(f"\n[BLOCKED/FILTERED] Dropped duplicate or system noise payload: {raw_id}")
        continue

    latest_norm_event = current_window['messages'][-1]
    parent_ref = f"(Reply to: {latest_norm_event['parent_id'][:8]}...)" if latest_norm_event.get('parent_id') else "(Root Post)"

    print(f"\n[EVENT INGESTION] Processed: {latest_norm_event['message_id']}... ({latest_norm_event['event_type']}) {parent_ref}")
    print(f" ├── Thread: {latest_norm_event['thread_id']} | Submolt: {latest_norm_event['submolt_id']} | Agent: @{latest_norm_event['agent_id']}")
    print(f" └── Content: \"{latest_norm_event['content'][:70].replace(chr(10), ' ')}...\"")
    print(f"[WINDOW MANAGER] Window v{current_window['version']} updated. ({len(current_window['messages'])} msgs in buffer)")

    if current_window.get("conversation_summary"):
        print(f" └── Rolling Summary: {current_window['conversation_summary']}")

    # S3: Discover / Correlate Topics - Semantic Topic Discovery & Cluster Correlation
    active_topics = discovery_engine.discover_topics(current_window, latest_norm_event)

    # State Sync: Feed discovered active topics back into Stage 2 Window state
    active_topic_ids = [t['topic_id'] for t in active_topics if t.get('topic_status') == 'ACTIVE']
    window_mgr.sync_topic_refs(current_window['thread_id'], active_topic_ids)

    # S4 & S5: Enrich & Score Active Topic Candidates
    latest_enriched_candidates = []
    latest_scored_candidates = []
    print("[STAGE 3, 4 & 5] Active Clusters, Enrichment & Actionability Scores:")
    for topic in active_topics:
        # if topic.get('topic_status') != 'ACTIVE':
        #     continue

        is_matched = latest_norm_event['message_id'] in topic.get('message_ids', [])
        match_flag = "[Matched]" if is_matched else "         "
        print(f" ├── {match_flag} Topic: {topic.get('topic_id')} -> ({len(topic.get('message_ids', []))} msgs) | Label: {topic.get('label')} | Status: {topic.get('topic_status')} | Timestamps: Seen={topic.get('first_seen_at')} | Updated={topic.get('last_updated_at')}")
        # S4: Enrich active cluster
        enriched_candidate = intel_engine.enrich_topic(topic, current_window)
        enriched_topic_lookup[topic.get('topic_id')] = enriched_candidate
        latest_enriched_candidates.append(enriched_candidate)

        if is_matched:
            print(f" │            Topic Label: {topic.get('label')}")
            print(f" │   └── [S4 ENRICHED] Title: {enriched_candidate['title']}")
            print(f" │       ├── Objectives ({len(enriched_candidate['objectives'])}): {enriched_candidate['objectives'][:2]}")
            print(f" │       ├── Hard Constraints: {enriched_candidate['hard_constraints'][:2]}")
            print(f" │       ├── Conflicts Detected: {len(enriched_candidate['conflicts'])}")
            print(f" │       └── Status: Owner={enriched_candidate['owner_status']} | Resolution={enriched_candidate['resolution_status']}")

        # S5: Score
        scored_candidate = intel_engine.score_candidate(enriched_candidate, threshold=0.60)
        latest_scored_candidates.append(scored_candidate)

        status_tag = f"[{scored_candidate['threshold_result']}]"
        print(f" │   └── {status_tag} Score: {scored_candidate['score']} (Threshold: {scored_candidate['threshold']})")
        print(f" │       ├── Title: {enriched_candidate['title']}")
        print(f" │       ├── Breakdown: {scored_candidate['score_breakdown']}")
        print(f" │       └── Reasons: {scored_candidate['reasons']}")

print("\n" + "=" * 80)
print(f"STREAM INGESTION & SCORING COMPLETE (Total Active scored candidates: {len(latest_scored_candidates)})")
print("=" * 80)

# Sample Inspection of the First Enriched Candidate
if latest_enriched_candidates:
    print("\n[SAMPLE ENRICHED CANDIDATE PAYLOAD READY FOR STAGE 5]:")
    print(json.dumps(latest_enriched_candidates[0], indent=2))

# -------------------------------------------------------------
# STAGE 6 & 7: RANK QUALIFIED TOPIC CANDIDATES AND USING GATEKEEPER
# -------------------------------------------------------------
# Segregate Qualified Candidates (to hand off to Stage 6) from Continued Monitoring
qualified_candidates = [sc for sc in latest_scored_candidates if sc.get("threshold_result") == "QUALIFIED"]
continue_monitoring = [sc for sc in latest_scored_candidates if sc.get("threshold_result") == "CONTINUE_MONITORING"]


print(f"\nSummary of Candidate Actionability Pipeline:")
print(f" • Qualified Candidates (Passing to Stage 6 Ranking): {len(qualified_candidates)}")
print(f" • Sub-threshold Candidates (Returned to Monitoring): {len(continue_monitoring)}")

if qualified_candidates:
    print("\n[SAMPLE QUALIFIED CANDIDATE READY FOR STAGE 6 RANKING]:")
    print(json.dumps(qualified_candidates[0], indent=2))

ranked_candidate_set = intel_engine.rank_candidates(
    qualified_candidates=qualified_candidates,
    thread_id=target_post_id
)

print(f"\n[STAGE 6]: CANDIDATE RANKING SUMMARY")
print(f" • Qualified Candidates Ranked : {len(ranked_candidate_set.get('candidates', []))}")
print(f" • Sub-threshold Candidates     : {len(continue_monitoring)} (Returned to monitoring)")

for candidate in ranked_candidate_set.get("candidates", []):
    print(f"\n 🏆 Rank {candidate['rank']} [{candidate['designation']}]: Topic ID '{candidate['topic_id']}'")
    print(f"    ├── Score: {candidate['score']}")
    print(f"    └── Title: {candidate['candidate_details']['enriched_topic']['title']}")

print(json.dumps(ranked_candidate_set, indent=2))

# -------------------------------------------------------------
# STAGE 7: DEDUPLICATE, SUPPRESS, MERGE, OR DEFER
# -------------------------------------------------------------
print("\n" + "=" * 80)
print("[STAGE 7]: DEDUPLICATION, SUPPRESSION & SAFETY CHECKS")
print("=" * 80)

stage_7_results = intel_engine.evaluate_candidate_dispositions(
    ranked_set=ranked_candidate_set,
    current_window=final_window or {},
    system_state=system_state_store
)

print("\n[CANDIDATE DISPOSITIONS AUDIT]:")
for disp in stage_7_results.get("dispositions", []):
    print(f" • Topic ID [{disp['topic_id']}]: {disp['disposition']}")
    print(f"   └── Reason: {disp['reason']}")

eligible_candidates = stage_7_results.get("eligible_candidates", [])
print(f"\nTotal Eligible Mission Candidates Approved: {len(eligible_candidates)}")

if eligible_candidates:
    print("\n" + "=" * 80)
    print("[STAGE 7: ELIGIBLE MISSION CANDIDATE PAYLOAD READY FOR STAGE 8 VALIDATION]:")
    print("=" * 80)
    print(json.dumps(eligible_candidates[0], indent=2))

# -------------------------------------------------------------
# STAGE 8-9: CANDIDATE VALIDATION NEGOTIATION
# -------------------------------------------------------------
print("\n" + "=" * 80)
print("[STAGE 8-9]: MULTI-AGENT CANDIDATE VALIDATION NEGOTIATION & GOVERNANCE GATE")
print("=" * 80)

validated_missions = []

for candidate in eligible_candidates:
    print(f"\nInitiating NegMAS validation mechanism for Candidate ID: [{candidate['candidate_id']}]...")
    print(f" • Topic: \"{candidate['title']}\"")
    print(f" • Actionability Score: {candidate['actionability_score']}")

    val_output = validation_engine.run_validation(candidate)
    val_res = val_output["validation_result"]
    audit_rec = val_output["audit_record"]

    print(f"\n[NEGMAS AGENT VOTING RESULTS]:")
    for vote in val_res["decisions"]:
        print(f" ├── @{vote['agent_id']}: [{vote['decision']}]")
        print(f" │   └── Rationale: \"{vote['rationale']}\"")

    print(f"\n[CONSENSUS OUTCOME]: {val_res['result']} (Consensus Reached: {val_res['consensus_reached']})")

    if val_res["result"] == "MISSION_NEEDED":
        candidate["status"] = "VALIDATED"
        validated_missions.append(candidate)

# -------------------------------------------------------------
# STAGE 9: GOVERNANCE GATE
# -------------------------------------------------------------
print("\n" + "=" * 80)
print("STAGE 9: GOVERNANCE GATE")
print("=" * 80)

authorized_missions = []
for mission in validated_missions:
    gov_decision = execute_governance_gate(mission, auto_approve=True)
    print(f"Governance Verdict for [{mission['candidate_id']}]: {gov_decision['governance_status']} ({gov_decision['decision_mode']})")
    print(f" ├── Rationale: \"{gov_decision.get('rationale') or gov_decision.get('reason')}\"")
    print(f" ├── Applied Policies: {gov_decision.get('policy_ids', [])}")
    print(f" └── Propagated Constraints: {gov_decision.get('constraints_to_propagate', [])}")

    if gov_decision["decision_mode"] == "AUTO_APPROVE":
        mission["governance"] = gov_decision
        authorized_missions.append(mission)

print(f"\nTotal Authorized Missions Ready for Downstream Execution (Stage 10/11): {len(authorized_missions)}")

if authorized_missions:
    print("\n" + "=" * 80)
    print("[FINAL VALIDATED & GOVERNED MISSION PAYLOAD]:")
    print("=" * 80)
    print(json.dumps(authorized_missions[0], indent=2))

# -------------------------------------------------------------
# STAGES 10-13: MYCELIUM HANDOFF
# -------------------------------------------------------------
print("\n" + "=" * 80)
print("[STAGES 10-13]: MISSION COMPILATION, NEGOTIATION, AND PUBLISHING")
print("=" * 80)

def convert_decimals_to_floats(obj):
    if isinstance(obj, decimal.Decimal): return float(obj)
    if isinstance(obj, dict): return {k: convert_decimals_to_floats(v) for k, v in obj.items()}
    if isinstance(obj, list): return [convert_decimals_to_floats(elem) for elem in obj]
    return obj


def compile_mission_specification(auth_mission: dict, enriched_topic: dict, room_name: str) -> dict:
    """
    Transforms the authorized candidate into a formal Mission schema.
    """
    now_epoch = int(datetime.now().timestamp())
    candidate_id = auth_mission.get("candidate_id", "cand-unknown")
    summary_text = auth_mission.get("summary", "").lower()

    # Extract domain constraints + propagated governance rules
    # base_constraints = [
    #     "Total 4-year cost for ASU must remain under $100,000",
    #     "Scholarship renewal requires maintaining a 3.0 GPA"
    # ]
    base_constraints = enriched_topic.get("hard_constraints", [])
    gov_constraints = auth_mission.get("governance", {}).get("constraints_to_propagate", [])
    all_constraints = base_constraints + gov_constraints

    trade_offs = []
    soft_prefs = enriched_topic.get("soft_preferences", [])

    # Soft Preferences vs. Hard Constraints
    if soft_prefs and base_constraints:
        trade_offs.append(f"Preference: [{soft_prefs[0]}] vs. Constraint: [{base_constraints[0]}]")
        if len(soft_prefs) > 1 and len(base_constraints) > 1:
            trade_offs.append(f"Preference: [{soft_prefs[1]}] vs. Constraint: [{base_constraints[1]}]")

    # Fallback to Unresolved Questions if Soft Preferences vs. Hard Constraints yields nothing
    unresolved_qs = enriched_topic.get("unresolved_questions", [])
    if not trade_offs and unresolved_qs:
        for q in unresolved_qs:
            trade_offs.append(f"Risk Evaluation: {q}")
    # Failsafe
    if not trade_offs:
        trade_offs = ["Balance financial risk against career and educational outcomes", "Scholarship Maintenance (3.0 GPA) vs. Career Flexibility"]

    return {
        "mission_id": f"mission-{candidate_id.replace('candidate-', '')}-{now_epoch}",
        "source_candidate_id": candidate_id,
        "mycelium_room_id": room_name,
        "objective": auth_mission.get("title"),
        "problem_statement": auth_mission.get("summary"),
        "participants": auth_mission.get("participant_ids", []),
        "status": "NEGOTIATION_PENDING",
        "constraints": all_constraints,
        "trade_offs": trade_offs,
        "created_at": datetime.now(timezone.utc).isoformat()
    }

# -------------------------------------------------------------
# STAGE 10: CREATE THE MISSION SPECIFICATION
# -------------------------------------------------------------
print("=" * 80)
print("[STAGE 10]: CREATE FORMAL MISSION SPECIFICATION")
print("=" * 80)

for auth_mission in authorized_missions:
  topic_id = auth_mission.get("source_topic_id")
  # Pull dynamic Stage 4 metadata via lookup map
  enriched_topic = enriched_topic_lookup.get(topic_id, {})

  # Build Formal Mission Specification
  formal_mission = compile_mission_specification(auth_mission, enriched_topic, cfg.mycelium_room)
  print(f"\n[STAGE 10] Mission Specification Compiled for {formal_mission['mission_id']}")
  print(json.dumps(formal_mission, indent=2))

  # -------------------------------------------------------------
  # STAGE 11: CONDUCT THE MISSION NEGOTIATION (MYCELIUM SESSION)
  # -------------------------------------------------------------
  # Build formatted constraints and trade-offs
  formatted_constraints = "\n".join(f"- {c}" for c in formal_mission['constraints'])
  formatted_trade_offs = "\n".join(f"- {t}" for t in formal_mission['trade_offs'])

  # Build Topic payload - Map formal mission into the format/ required schema expected by bridge.trigger_mycelium()
  handoff_topic = {
        "id": target_post_id,
        "title": formal_mission["objective"],
        "content": f"Problem:\n{formal_mission['problem_statement']}\n\nKnown Constraints:\n{formatted_constraints}\n\nIdentified Trade-offs:\n{formatted_trade_offs}"
    }

  # Capture cost and token spent baseline before running negotiation
  baseline_snapshot = bridge.get_proxy_snapshot()

  print(f"[STAGE 11] Triggering Mycelium coordination session in room: [{cfg.mycelium_room}]...")
  session_meta = bridge.trigger_mycelium(handoff_topic)
  print(f" • Coordination Session ID: {session_meta['coordination_session_id']}")

  # Post kickoff acknowledgment comment on Moltbook thread
  if not USE_LOCAL_JSON:
    try:
      bridge.moltbook.comment_on_post(target_post_id, "AI negotiation started via orchestrated mission framework...")
      print(" • Kickoff comment published to Moltbook.")
    except Exception as e:
        print(f" • [WARNING] Failed to post kickoff comment. Proceeding with negotiation anyway. Error: {e}")

  print("\nPolling Mycelium for negotiation convergence...")
  negotiation_payload = bridge.poll_for_summary(handoff_topic, target_post_id)
  # Attach baseline snapshot regardless of outcome to guarantee spend calculation
  # Convert Decimal objects to float before attaching to negotiation_payload
  negotiation_payload["baseline_snapshot"] = convert_decimals_to_floats(baseline_snapshot)
  session_status = negotiation_payload.get("status")

  if session_status != "session_complete":
    print(f"\n⚠️ [NEGOTIATION NON-CONVERGENCE] Mycelium negotiation did not complete successfully - it failed or timed out. Session ended with status: {session_status}")
    print("Skipping Moltbook public consensus publication per governance rules.")

    # Record failed state in bridge state store
    bridge.state.mark_topic(target_post_id, status=session_status, final_summary=None, session_result=negotiation_payload)

    # Execute token & cost audit to account for spent resources during negotiation
    bridge.print_token_audit(baseline_snapshot=baseline_snapshot)
  else:
      print("\n[MYCELIUM NEGOTIATION COMPLETE] Session converged successfully.")

      # -------------------------------------------------------------
      # STAGE MISSION PLAN GENERATION & MOLTBOOK PUBLISHING
      # -------------------------------------------------------------
      print("\n" + "=" * 80)
      print("[STAGE 13]: MISSION PLAN GENERATION & MOLTBOOK PUBLISHING")
      print("=" * 80)
      if not USE_LOCAL_JSON:
        try:
            # Publish root consensus outcome and nested agent replies back to Moltbook
            final_summary = bridge.publish_summarized_result(target_post_id, negotiation_payload)
        except Exception as e:
            print(f"⚠️ Failed to publish summary to Moltbook: {e}")
      else:
            print(" Demo Mode: Skipping live Moltbook write-back. Pipeline succeeded.")


print("\n" + "="*80)
print("ATM Pipeline Stages 1-13 Completed.")
print("=" * 80)